In [ ]:
import yfinance as yf
import numpy as np 
import pandas as pd 
import datetime
from spark_session import get_spark
from pyspark.sql import functions as F
from delta.tables import DeltaTable
TICKERS = {
    'GC=F':  'Gold',
    'SI=F':  'Silver',
    'PL=F':  'Platinum',
    'PA=F':  'Palladium',
    'HG=F':  'Copper',
    'CL=F':  'WTI Crude Oil',
    'BZ=F':  'Brent Crude',
    'XEL':   'Xcel Energy',
    'CVX':   'Chevron',
    'BAC':    'Bank of America',
    'BAH':     'Booze Allen Hamilton'
}
spark = get_spark("stocks")
#grab yfinance data 
ticker_lst =['PL=F', 'GC=F','SI=F','HG=F','PA=F', 'CL=F', 'BZ=F', 'XEL', 'CVX', 'BAC', 'BAH']
dt = yf.download(ticker_lst, start='2020-01-01', group_by='ticker')
#Download historical data for the last year
dt = pd.DataFrame(data=dt)
dt_f = dt.reset_index()
dt_f.columns = ['_'.join(col).strip() for col in dt_f.columns.values] #transform white space to underscore.
dt_f.columns = ["".join(col).replace('=','_') for col in dt_f.columns.values] #change '=' to underscore.
dt_f = np.round(dt_f, decimals=2)
        #print(dt_f.columns)

df_spark = spark.createDataFrame(dt_f)
df_spark = df_spark.withColumn('DateKey', F.date_format(F.col('Date_'),'yyyyMMdd').cast("int"))
df_spark = df_spark.withColumn('Date_',F.date_format(F.col('Date_'),'yyyy-MM-dd'))
df_spark.printSchema()
df_spark.withColumn("Yr", F.year(F.col("Date_")))
#load into delta table
df_spark.write.mode("overwrite").format("delta").option("inferSchema","true").saveAsTable("stocks")

In [1]:
import yfinance as yf
import numpy as np 
import pandas as pd 
import datetime
from spark_session import get_spark
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = get_spark('merge')
df = DeltaTable.forName(spark, 'stocks')
df_max  =df.toDF()
df_max.select(F.max(F.col("Date_"))).show()
df_max_dt = df_max.select(F.max)

AnalysisException: org.apache.hadoop.hive.ql.metadata.HiveException: java.lang.RuntimeException: Unable to instantiate org.apache.hadoop.hive.ql.metadata.SessionHiveMetaStoreClient